# Versione part 1 Marghe pulita 

## 1) Structural preprocessing: Apply the following steps using the methods of your choice (e.g., fMRIPrep, FSL-based, python-based):

a) Skull-stripping. Report and plot the original T1w image and the skull-stripped brain to visually assess the quality of the brain extraction.

b) Segmentation. Report and plot the segmented tissues on the brain.

c) Registration. Register your results bringing them to MNI space.
Report a plot of the T1 alignment with MNI before and after this step.

In [1]:
## 0. imports
import fsl.wrappers
from fsl.wrappers import fslmaths
from fsl.wrappers import fslroi, mcflirt

import mne_nirs
import nilearn
from nilearn.datasets import fetch_development_fmri

import mne
import mne_nirs
import dipy
from dipy.data import fetch_bundles_2_subjects, read_bundles_2_subjects
import xml.etree.ElementTree as ET
import os.path as op
import nibabel as nib
import glob

#!pip install antspyx
#import ants

import openneuro
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, print_dir_tree, make_report


import requests
import urllib.request
from tqdm import tqdm


from fsl.wrappers import fast, bet
from fsl.wrappers.misc import fslroi
from fsl.wrappers import flirt

from mne_bids import print_dir_tree


import glob
import pandas as pd
import numpy as np
import json
import subprocess

import sys
import os
import glob

notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
sys.path.append('.')

import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
await lmod.load('freesurfer/7.4.1')
await lmod.list()

from utils import loadFSL, FSLeyesServer, mkdir_no_exist, interactive_MCQ

import threading


import os.path as op


import matplotlib.pyplot as plt

### 0.1 importare data

In [2]:
# project directories
#data_dir =  "/home/jovyan/data/tsclient/NSSP/progetto"
#output_dir = "/home/jovyan/data/tsclient/NSSP/progetto/outputs/structural"

# Path per Luca (sono pigro e non avevo voglia di cambiare tutte le cartelle)
data_dir = "/home/jovyan/data/tsclient/Projects/Project_1/miniproject_2026_final"
output_dir = "/home/jovyan/data/tsclient/Projects/Project_1/outputs"

print("Notebook directory:", notebook_dir)
print("Project directory:", parent_dir)
print("Data directory:", data_dir)
print("Output directory:", output_dir)

os.environ["DIPY_HOME"] = "/home/jovyan/tsclient/dipy_data"


print(os.path.isdir(data_dir))
print(os.listdir(data_dir))

Notebook directory: /data/Desktop/myfiles/NSSP_project_1/notebooks
Project directory: /data/Desktop/myfiles/NSSP_project_1
Data directory: /home/jovyan/data/tsclient/Projects/Project_1/miniproject_2026_final
Output directory: /home/jovyan/data/tsclient/Projects/Project_1/outputs
True
['dataset_description.md', 'fMRI', 'T1']


### 0.2 start FSL

In [3]:
# setup FSL path
loadFSL()

In [4]:
%gui wx


(ipykernel_launcher.py:2037): dbind-WARNING **: 14:21:42.194: Couldn't connect to accessibility bus: Failed to connect to socket /run/user/911245073/at-spi/bus_100: No such file or directory


In [5]:
# Start FSLeyes 
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

14:21:46: Debug: Adding duplicate image handler for 'Windows bitmap file'
14:21:46: Debug: Adding duplicate animation handler for '1' type
14:21:46: Debug: Adding duplicate animation handler for '2' type
14:21:46: Debug: Adding duplicate image handler for 'Windows bitmap file'
14:21:46: Debug: Adding duplicate animation handler for '1' type
14:21:46: Debug: Adding duplicate animation handler for '2' type

(ipykernel_launcher.py:2037): Gtk-CRITICAL **: 14:21:46.379: gtk_window_resize: assertion 'width > 0' failed

(ipykernel_launcher.py:2037): Gtk-CRITICAL **: 14:21:48.388: gtk_window_resize: assertion 'width > 0' failed


### 0.3 load images per structural preprocessing 

In [6]:
# Search recursively for T1 images: check for all the files containing T1 and that finish with .nii.gz
# Magari non è necessario?
t1_files = glob.glob(
    os.path.join(data_dir, "**", "*T1*.nii.gz"),
    recursive=True
)

# Al posto della recursively possiamo mettere direttamente questo 
# t1_files = "/home/jovyan/data/tsclient/NSSP/progetto/T1-20261015T140821Z-1-001/T1/T1w.nii.gz"
# Questo non funziona perché così t1_files è una stringa e t1_path diventa semplciemente "/"

print("T1 images found:")
print(t1_files)


# Check input
if len(t1_files) != 1:
    raise ValueError(f"Expected one T1w image, found: {t1_files}")

t1_path = t1_files[0]

# Create output folder
os.makedirs(output_dir, exist_ok=True)

# Verify paths
print("Project exists:", os.path.isdir(parent_dir))
print("Data exists:", os.path.isdir(data_dir))
print("Input T1w:", t1_path)
print("Output directory:", output_dir)

T1 images found:
['/home/jovyan/data/tsclient/Projects/Project_1/miniproject_2026_final/T1/T1w.nii.gz']
Project exists: True
Data exists: True
Input T1w: /home/jovyan/data/tsclient/Projects/Project_1/miniproject_2026_final/T1/T1w.nii.gz
Output directory: /home/jovyan/data/tsclient/Projects/Project_1/outputs


## 1 Anatomical preprocessing

Display image

In [7]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_path) # load the T1w image


(ipykernel_launcher.py:2037): Gtk-WARNING **: 14:22:57.952: Negative content width -11 (allocation 9, extents 10x10) while allocating gadget (node button, owner GtkButton)

(ipykernel_launcher.py:2037): Gtk-WARNING **: 14:22:57.952: Negative content height -2 (allocation 8, extents 5x5) while allocating gadget (node button, owner GtkButton)

(ipykernel_launcher.py:2037): Gtk-WARNING **: 14:22:57.960: Negative content width -2 (allocation 18, extents 10x10) while allocating gadget (node button, owner GtkToggleButton)


In [9]:
### 1a Skull stripping 
def get_skull_stripped_anatomical(t1_path, output_dir, robust=False):
    """
    Function to perform skull-stripping (removing the skull around the brain).
    This is a simple wrapper around the brain extraction tool (BET) in FSL's suite
    It assumes data to be in the BIDS format (which we will cover in the following weeks).
    The method also saves the brain mask which was used to extract the brain.

    The brain extraction is conducted only on the T1w of the participant.

    Parameters
    ----------
    t1_path: string
        Path to the original T1w image.
    output_dir: string
        Directory where preprocessing results are saved.
    robust: bool
        Whether to conduct robust center estimation with BET or not. Default is False.

    Outputs
    -------
    T1w_brian.nii.gz 
        skull-stripped anatomical image
    T1w_brain_mask.nii.gz 
        binary brain mask
    """
    # output directory 
    os.makedirs(output_dir, exist_ok= True)

    # output path
    betted_brain_path = os.path.join(output_dir, "T1w_brain.nii.gz")
    
    # run BET
    os.system('bet {} {} -m {}'.format(t1_path, betted_brain_path, '-R' if robust else ''))
    print("Done with BET.")

# resulting mask path
resulting_mask_path = os.path.join(output_dir, "T1w_brain_mask.nii.gz")

# perform skull stripping 
get_skull_stripped_anatomical(t1_path, output_dir)

# visualize 
fsleyesDisplay.load(resulting_mask_path)

/opt/fsl-6.0.7.18/bin/bet failed during command:/home/jovyan/data/tsclient/Projects/Project_1/miniproject_2026_final/T1/T1w.nii.gz /home/jovyan/data/tsclient/Projects/Project_1/outputs/T1w_brain.nii.gz -m
Done with BET.


terminate called after throwing an instance of 'NiftiIO::NiftiException'
  what():  Error: cant open file /home/jovyan/data/tsclient/Projects/Project_1/outputs/T1w_brain.nii.gz
/opt/fsl-6.0.7.18/bin/bet: line 399:  2316 Aborted                 (core dumped) ${FSLDIR}/bin/bet2 $IN $OUT $bet2opts


PathError: Could not find a supported file with prefix "/data/tsclient/Projects/Project_1/outputs/T1w_brain_mask.nii.gz"

In [ ]:
# visualize the skull-stripped brain
brain_path = os.path.join(output_dir, "T1w_brain.nii.gz")
# load skull-stripped brain
fsleyesDisplay.load(brain_path)

In [ ]:
# poi potremmo migliorare il fit però a me non sembra necessario
#get_skull_stripped_anatomical(bids_root, preproc_root, "001", robust=True)
#fsleyesDisplay.resetOverlays()
#fsleyesDisplay.load(op.join(bids_root, 'sub-001', 'anat', 'sub-001_T1w.nii.gz'))
#fsleyesDisplay.load(resulting_mask_path)


In [ ]:
# compare the different images 
anatomical_path = t1_path # original T1w
resulting_brain_mask= os.path.join(output_dir, "T1w_brain_mask.nii.gz") # binary mask
betted_brain_path = os.path.join(output_dir, "T1w_brain.nii.gz") # skull-tripped T1
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(anatomical_path)
fsleyesDisplay.load(betted_brain_path)
fsleyesDisplay.load(resulting_brain_mask)

Per report  
Skull stripping was performed on the original T1-weighted anatonmical image using BET, with default parameters.  
This steps enables to remove non-brain tissues, such as the skull and scalp, while preserving brain structures for subsequent preprocessing.  
BET generated a skull-stripped T1w image and a binary brain mask. The quality of this extraction was visually good so no further mask editing was performed.  
From the two images below is possible to visually compare and inspect the result of this first step and to conclude that the resulting mask successfully excluded the non-brain tissues.   

Mettere qui sotto immagine pre e post skull stripping.


### 1b Tissue segmentation

In [ ]:

bet_path = os.path.join(output_dir, "T1w_brain.nii.gz")


fast_target = bet_path # segmentation is applied to the skull-stripped image 

# [os.remove(f) for f in glob.glob(op.join(output_dir, "T1w_fast*"))] # Just to clean the directory in between runs of the cell

segmentation_path = op.join(output_dir, "T1w_fast")

fast(imgs=[fast_target], out=segmentation_path, n_classes=3)

print_dir_tree(output_dir, max_depth=5)
print("FAST segmentation completed.")

# Saving images for the report??

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(bet_path)

csf_path = op.join(output_dir, "T1w_fast_pve_0.nii.gz")
gm_path = op.join(output_dir, "T1w_fast_pve_1.nii.gz")
wm_path = op.join(output_dir, "T1w_fast_pve_2.nii.gz")

fsleyesDisplay.load(csf_path)
fsleyesDisplay.load(gm_path)
fsleyesDisplay.load(wm_path)
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'# CSF
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green' # GM
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue' #WM

Per report  
Tissue segmentation was performe on the skull-stripped T1-weighted image using FSL FAST, which classifies brain tissue into three classes: white matter (WM), gray matter (GM) and cerebrospinal fluid (CSF).   
To generate this tissue-specific maps, FAST classifies the voxels based on their signal intensity and spatial neighbourhood, while correcting for intensity inhomogeneities. 
These maps were visualized as coored overlays on the anatomical image, where CSF is in red, GM in green and WM in blue.  

### 1c Registration

Registration. Register your results bringing them to MNI space. 
Report a plot of the T1 alignment with MNI before and after this step.   

so the steps are:  
- register the anatomical T1 in MNI space  
- visualize the alignment before the registration   
- visualize the alignment after the registration  
- comment the quality of the alignment 

FLIRT (FMRIB's linear image registration tool) computes a spatial transformation to align the two images.  
- source = subject's skull tripped T1w  
- reference = MNI152 T1 1 mm brain template

In [ ]:
#Please note that the values in the transformation matrix of flirt refer to:
#Top-left 3×3 block: rotation, scaling, shearing.
#Last column: translation in x, y, z.
#Bottom row: always [0 0 0 1] for affine transforms.

In [ ]:
# subject anatomical image (skull-stripped)
subject_anatomical =op.join(output_dir, "T1w_brain")
source = subject_anatomical

# MNI152 standard brain template
mni_template = op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm_brain'))
reference = mni_template

print("Source (subject T1):", source)
print("Reference (MNI152):", reference) 

In [ ]:
# Before registration: visual quality control
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference) 
fsleyesDisplay.load(source)

Affine (linear) registration using FSL FLIRT. 

In [ ]:
# T1 --> MNI affine registration 
# FLIRT estimatea a 12-degree-of-freedom affine transformation: 3 translations, 3 rotations, 3 scaling ans 3 range components.  
# we compare two cost functions:
# 1) leastsq: minimizes squared intensity differences
# 2) normcorr: evaluates correlation between intensity patterns and is less sensitive to global intensity scaling 
# the rsulting images and transformation matrices are saved separately so that the registration with the best anatomical alignment cna be selected for subsequent analyses

result = op.join(output_dir, "T1w_mni")

# So FSL FLIRT offers several cost functions for image registration. 
# Mutual information measures statistical dependence between image intensity distributions. 
# Correlation ratio evaluates how well one image can predict another using conditional variance statistics. 
# Normalized correlation measures linear correlation between image intensities using the Pearson correlation coefficient. 
# Normalized mutual information provides a scale-independent version of mutual information. 
# Least squares minimizes squared intensity differences by assuming corresponding voxels should have similar intensities. 
# Label difference minimizes mismatched labels for discrete segmentation images. 
# Each function makes different assumptions about the relationship between corresponding image intensities.

# possible_costs = ['mutualinfo', 'corratio', 'normcorr', 'normmi', 'leastsq', 'labeldiff']

# We consider only within-modalities costs, as the two images belong to the same modality: least squares and normalized correlation ratio.
costs_to_consider = [ 'leastsq', 'normcorr' ] 

for c in costs_to_consider:
    # output path for the registered anatomical image
    registered_image = result + '_' + c

    # output path for the corresponding affine transformation matrix
    transformation_matrix = op.join(output_dir, 'T1w_to_MNI_' + c + '.mat')

    # run FLIRT regirstration
    flirt(source, reference, out=registered_image, omat=transformation_matrix, dof=12, cost= c)

    print(f"Registration completed usign {c}")
    print(f"Registred image:{registered_image}")
    print(f"Tranformation matri: {transformation_matrix}")





In [ ]:
# After registration: visual quality control 

# least squares
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference)
fsleyesDisplay.load(result + '_leastsq') # subject T1 registered using least squares


# normalized correlation
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference)
fsleyesDisplay.load(result + '_normcorr') # subject T1 registered using normalized correlation

In [ ]:
# select final registration and verify outputs
# select the final registration base on visual quality contro

selected_cost = "normcorr" # oppure l'altro, non ho runnato, da vedere

# final registered T1w image in MNI space
registered_t1 = result + '_' + selected_cost

# final affine transformation matrix (T1 --> MNI)
t1_to_mni_mat = op.join(output_dir, "T1w_to_MNI_" + selected_cost + ".mat")

print("Selected cost function:", selected_cost)
print("Registered T1 exists:", op.isfile(registered_t1 + '.nii.gz'))
print("T1-to-MNI matrix exists:", op.isfile(t1_to_mni_mat))

# visualize 
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference)
fsleyesDisplay.load(registered_t1)

So far: EPI -.> FLIRT --> T1 --> FLIRT --> MNI

Nonlinear registration using FSL FNIRT.  
The difference is that FLIRT corrects the global differences while FNIRT also corrects the local anatomical differences.  

T1 aligned globally to MNI (Affine transformation matrix) --> FNIRT --> T1 normalized to MNI space (Affine + non linear deformation)

In [ ]:
# Original T1w for nonlinear registration 
original_t1 = t1_path

# MNI152 templates at 2 mm resolution
fsl_standard = op.expandvars("$FSLDIR/data/standard") # for FNIRT 2mm is better
reference_full = op.join(fsl_standard, "MNI152_T1_2mm")

registered_t1_nonlin = op.join(output_dir, "T1w_mni_nonlin.nii.gz") # T1 normalized using affine and no linear transformations

t1_to_mni_warp = op.join(output_dir, "T1w_to_MNI_warpcoef.nii.gz")

subprocess.run({"fnirt", "--in=" + original_t1,"--aff=" + t1_to_mni_mat,"--ref=" + reference_full, "--config=T1_2_MNI152_2mm","--iout=" + registered_t1_nonlin, "--cout=" + t1_to_mni_warp})

print("FNIRT nonlinear registration completed")


In [ ]:
# visual quality control
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference_full) # full MNI template
fsleyesDisplay.load(registered_t1_nonlin) # T1 registred using FLIRT + FNIRT 


Now: EPI --> affine --> T1 --> affine+nonlinear --> MNI

Per report:  

The T1-weighted anatomical image was normalized to the MNI152 standard space using a two-stage registration procedure implemented in FSL.

Forst, an affine registration was performed using FLIRT with 12 degrees of freedom to correct global differences in translation, rotation, scaling, and shearing.  
Two within-modality cost functions were tested: least squares, which minimizes squared differences between corresponding voxel intensities, and normalized correlation, which evaluated the similarity of intensity patterns and is less sensitive to gliba intensity scaling.  
Registration quality was assessed by compating the anatomical alignment with the MNI template before and after registration.  
The registration obtained using ____ cost function provided better anatomical correspondance and was therefore selected for subsequent preprocessing. 

The resulting affine transformation was then used to initialize nonlinear registration using FNIRT, applied to the original non-sull-stripped T1w image. this step account for local anatomical differences between the individual brain and the MNI template that cannot be corrected by affine registration alone.   

Registration quality was assessed by comparing the T1w image with the MNI template before registration, after affine registration and after nonlinear registration.  

Poi dobbiamo dire se ci sono differenze tra affine and no linear registration. 

come immagini possiamo bettere quella native T1 + MNI, affine alignment and nonlinear alignment.

## 2 Functional preprocessing

### 2a motion correction 

In [ ]:
fmri_path = os.path.join(data_dir, "fMRI-20261015T140819Z-1-001", "fMRI", "tfMRI_LANGUAGE_LR", "tfMRI_LANGUAGE_LR.nii.gz")

print("Functional MRI exists:", os.path.isfile(fmri_path))
print("Functional MRI path:", fmri_path)
      
# functional preprocessing output directory
func_output_dir = os.path.join(data_dir, "outputs", "functional")
os.makedirs(func_output_dir, exist_ok= True)

fmri_img = nib.load(fmri_path)

In [ ]:
mean_signal = fmri_img.get_fdata().mean(axis=(0,1,2))

plt.figure(figsize=(10,4))
plt.plot(mean_signal)
plt.xlabel('Time (volume)')
plt.ylabel('Mean voxel intensity')
plt.title("Global mean fMRI signal before volume removal")
plt.show()


In [ ]:
file_to_trim = fmri_path
# output functional directory 
func_output_dir = op.join(data_dir, 'outputs', 'functional')
mkdir_no_exist(func_output_dir)

# output image
output_target = op.join(func_output_dir, 'tfMRI_LANGUAGE_LR_settled.nii.gz')

# total numeber of volumes
total_volumes = nib.load(file_to_trim).shape[-1]
# remove first volumes --> dall'immagine sopra vediamo che il volume più problematico è il primo 

start_vol = 1 # Where should we start? (First volume is 0, not 1 !)
number_of_volumes = total_volumes - start_vol # How many volumes should we keep?

# Function creating new nifti with trimmed number of volumes
fslroi(file_to_trim, output_target, str(start_vol), str(number_of_volumes))
print(f"You kept {nib.load(output_target).shape[-1]} volumes.")

In [ ]:
# Mean voxel intensity after removing the first volume
settled_img = nib.load(output_target)
mean_signal_settled = settled_img.get_fdata().mean(axis=(0, 1, 2))

plt.figure(figsize=(10, 4))
plt.plot(mean_signal_settled)
plt.xlabel('Time (volume)')
plt.ylabel('Mean voxel intensity')
plt.title("Global mean fMRI signal after volume removal")
plt.show()

In [ ]:
# Perform rigid-body motion correction using FSL MCFLIRT.
# MCFLIRT estimates 6 motion parameters:
# - 3 rotations (x, y, z)
# - 3 translations (x, y, z)
# Each functional volume is aligned to a reference volume to reduce spatial inconsistencies caused by head movement.

# input fMRI after volume removal 
path_original_data = output_target

# output motion-corrected fMRI
path_moco_data = op.join(func_output_dir, "tfMRI_LANGUAGE_LR_moco")

# motion correction
mcflirt(infile=path_original_data,o=path_moco_data, plots=True, report=True, dof=6, mats=True)
print("Morion correction completed")

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(path_original_data)
fsleyesDisplay.load(path_moco_data)

In [ ]:
def load_mot_params_fsl_6_dof(path):
    return pd.read_csv(path, sep='  ', header=None, 
            engine='python', names=['Rotation x', 'Rotation y', 'Rotation z','Translation x', 'Translation y', 'Translation z'])

mot_params = load_mot_params_fsl_6_dof(path_moco_data + ".par")
print("Motion parameters shape:", mot_params.shape)
mot_params

In [ ]:
# Compute Power framewise displacement (FD).
# FD measures frame-to-frame head movement using all six motion parameters estimated by MCFLIRT.
# Rotations are converted from radians to millimetres assuming a spherical head radius of 50 mm.

def compute_FD_power(mot_params):

    # difference between consecutive volumes
    framewise_diff = mot_params.diff().iloc[1:]

    # rotational displacement (radiants)
    rot_params = framewise_diff[['Rotation x', 'Rotation y', 'Rotation z']]

    # convert rotations to equivalent displacement (mm)
    converted_rots = rot_params*50

    # total displacement (mm)
    trans_params = framewise_diff[['Translation x', 'Translation y', 'Translation z']]

    # sum of absolute rotational and translational differences
    fd = converted_rots.abs().sum(axis=1) + trans_params.abs().sum(axis=1)
    return fd

fd = compute_FD_power(mot_params).to_numpy()

# idnetify unusually high motion 
threshold = np.quantile(fd,0.75) + 1.5*(np.quantile(fd,0.75) - np.quantile(fd,0.25))

# Indices of FD values exceeding the threshold
outlier_indices = np.where(fd > threshold)[0]
# FD[0] corresponds to the movement between volumes 0 and 1 of the trimmed fMRI series.
# Therefore, +1 gives the index of the later volume.
outlier_volumes = outlier_indices + 1

#%matplotlib inline
# Plot FD across the scan
plt.figure(figsize=(10, 4))
plt.plot(list(range(1, fd.size+1)), fd)
plt.xlabel('Volume index after initial volume removal')
plt.ylabel('FD displacement (mm)')
plt.title("Framewise displacement after motion correction")
plt.axhline(y=threshold,color="black",linestyle="--",label=f"IQR threshold = {threshold:.3f} mm")
plt.legend()
plt.show()

# Report summary statistics
print(f"Mean FD: {np.mean(fd):.4f} mm")
print(f"Median FD: {np.median(fd):.4f} mm")
print(f"Maximum FD: {np.max(fd):.4f} mm")
print(f"IQR threshold: {threshold:.4f} mm")

print("Outlier volume indices:", outlier_volumes.tolist())
print(f"Outliers: {len(outlier_volumes)} / {len(fd)} FD values")

# SALVARE GLI INDICI PERCHè CREDO SERVANO DOPO NEL GLM
np.savetxt(op.join(func_output_dir, "fd_outlier_volumes.txt"), outlier_volumes, fmt="%d")

Per report  
Before motion correction, the global mean fMRI signal was inspected across volumes to identify irregular acquiaition. The first volume was removed using FSLROI because ti showed an unusual high mena intensity compared to the others, potentially reflecting incomplete signal stabilization.  
Motion correction was then performed using FSL MCFLIRT, which estimates six rigid-body motion parameters (3 translations and 3 rotations) to spatially align the funcitonal volumes and reduce motion-related inconsistencies. 
Head motion was quantified using Power's framewise displacement (FD), calculated as the sum of absolute frame-to-frame differences in the six motion parameters. Rotations were converted from rafians to mm assuming a 50mm head radius. Volumes with unusually high FD were identified using a data-driven threshold defined as Q3 + 1.5 * IQR.  

come figura mettiamo il FD   
Head motion during fMRI acquisition. Framewise displacement (FD) calculated from the six motion parameters estimated using FSL MCFLIRT. The dashed horizontal line indicates the data-driven outlier threshold defined as Q3 + 1.5 × IQR. Values above this threshold identify frames with unusually high estimated movement.

### 2b Gaussian smoothing 

- apply a gaussian smoothing filter to fMRI data after motion correction   
- chose an appropriate FWHM and justify it  
- extract the same volume before and after smoothing  
- visualize and compare the results  
- comment the effect of the filter  


The gaussian smoothing substitutes the intensity of each voxel with a weighted mean of the intensitied of nearby voxels, usign a gaissian kernel. Closer vocels receiv  higher weight, and further ones receive a progressively smaller weight.   
The main goal of this steps is to improve the signal-to-noise ration and reduce the spatial variations at high intensity.  

FWHM = Full width at half maximum and it's the width of the gaussian kernel measured at 50% of its maximum height.  
FLS fslmaths -s requires the standard deviation and not FWHM so: standard deviation = FWHM/2.3548.  
We chosse FHMW = 6mm because ____ --> chiedere ad assistenti se c'è una regola

In [ ]:
# to see which fwhm to chose
# Inspect functional voxel dimensions
fmri_voxel_size = nib.load(path_moco_data + ".nii.gz").header.get_zooms()[:3]

print("Functional voxel size (mm):", fmri_voxel_size)

fwhm = 6.0
sigma = fwhm/2.3548

# imput: motion-corrected fMRI
# output: smoosmoothed_path = op.join(
smoothed_path= op.join(func_output_dir,"tfMRI_LANGUAGE_LR_moco_smoothed-6mm")

subprocess.run(['fslmaths',path_moco_data, '-s', str(sigma), smoothed_path])

print("Smoothing completed")
print("FWHM:", fwhm, "mm")
print("Sigma:", round(sigma, 3), "mm")


In [ ]:
# select one fMRI volume
vol = 100

# before smoothing
before_smoothing = op.join(func_output_dir, "volume_before_smoothing")
fslroi(path_moco_data, before_smoothing, str(vol), str(1))

# after
after_smoothing = op.join(func_output_dir, "volume_after_smoothing")
fslroi(smoothed_path, after_smoothing, str(vol), str(1))

print("Selected volume:", vol)
print("Before and after smoothing volumes extracted.")

In [ ]:
# visualize before and after smoothing
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(before_smoothing)
fsleyesDisplay.load(after_smoothing)

Per report  
Gaussian spatial smoothing was appied to the motion-corrected fMRI data using FSL fslmaths, with a kernel of 6mm FWHM, corresponding to a standard deviation of approximately 2.55 mm.

A 6 mm FWHM was selected as a compromise between improving the signal-to-noise ratio (SNR) and preserving spatial specificity. Smoothing reduces high-frequency spatial noise by averaging neighbouring voxel intensities, but may also blur fine activation patterns.  

To asses its effect, the same functional volume (volume 100) was visualized before and aftre smoothing.  

mettere immagine e sotto questo commento:  
The smoothed image showed reduced voxel-to-voxel intensity variations and more homogeneous signal patterns, accompanied by slightly blurred anatomical boundaries, consistent with the expected effects of Gaussian smoothing.  


come figure legend possiamo mettere: Gaussian smoothing of functional MRI data. Comparison of the same functional volume (volume 100) before (A) and after (B) Gaussian smoothing using a 6 mm FWHM kernel.

### 2c Validate the alignment between anatomical and functional data

In [ ]:
# Input: motion-corrected functional data
epi_source = path_moco_data

# whole and skull-stripped structural images 
whole_t1 = t1_path
skull_stripped_t1 = op.join(output_dir, "T1w_brain.nii.gz")

# output for EPI-to-T1 registration 
output_path = op.join(func_output_dir, "tfMRI_LANGUAGE_LR_moco_bbr")

# extract a representative functional volume 
ref_vol_name =  op.join(func_output_dir, "tfMRI_LANGUAGE_LR_moco_vol_middle")

use_single_vol = True

if use_single_vol:
    # Extract the middle volume with fslroi 
    n_volumes = nib.load(epi_source + ".nii.gz").shape[-1]
    middle_vol = n_volumes // 2
    fslroi(epi_source, ref_vol_name, str(middle_vol), str(1))
    print("Selected EPI volume:", middle_vol)

In [ ]:
# Inspect EPI and MNI image geometry

epi_img = nib.load(ref_vol_name + ".nii.gz")
mni_img = nib.load(registered_t1_nonlin)

print("EPI shape:", epi_img.shape)
print("EPI voxel size:", epi_img.header.get_zooms()[:3])
print("EPI affine:")
print(epi_img.affine)

print("\nRegistered T1 shape:", mni_img.shape)
print("Registered T1 voxel size:", mni_img.header.get_zooms()[:3])
print("Registered T1 affine:")
print(mni_img.affine)

# inspect native EPI against the T1 normalized to MNI 
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(registered_t1_nonlin)
fsleyesDisplay.load(ref_vol_name)

In [ ]:
# if the alignment is already good, we can skip this cell

# if the initial alignment is not good we register the functional EPI to the subject's anatomical T1 
# White matter partial-volume map from FAST (Part 1b)
wm_pve_path = op.join(output_dir, "T1w_fast_pve_2.nii.gz")
white_matter_segmentation = op.join(output_dir, "T1w_fast_pve_2.nii.gz") # create a binary white matter segmentation or epi_reg
subprocess.run(["fslmaths",wm_pve_path,"-thr", "0.5","-bin", white_matter_segmentation])

# register EPI to anatomical T1
subprocess.run(['epi_reg',"--epi=" + ref_vol_name, "--t1=" + whole_t1, "--t1brain=" + skull_stripped_t1, "--out=" + output_path, "--wmseg=" + white_matter_segmentation])

# EPI-to-T1 transformation matrix
epi_to_t1_mat = output_path + ".mat"

print("EPI-to-T1 matrix exists:", op.isfile(epi_to_t1_mat))

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(skull_stripped_t1)
fsleyesDisplay.load(output_path)

In [ ]:
# now we combine:
# EPI --> T1 (the affine matrix)
# T1 --> MNI ( the non linear transformation obtained with FNIRT)

epi_mni_path = op.join(func_output_dir,"tfMRI_LANGUAGE_LR_vol_MNI.nii.gz")

# MNI reference used in Part 1c with FNIRT
mni_reference = reference_full

# Verify the required transformations
assert op.isfile(epi_to_t1_mat), "EPI-to-T1 matrix missing"
assert op.isfile(t1_to_mni_warp), "FNIRT warp missing"

# Apply combined transformation
subprocess.run(["applywarp", "--in=" + ref_vol_name,"--ref=" + mni_reference,"--warp=" + t1_to_mni_warp, "--premat=" + epi_to_t1_mat, "--out=" + epi_mni_path])

print("EPI successfully transformed to MNI space.")
print("Output exists:", op.isfile(epi_mni_path))

In [ ]:
# quality control: anatomical-functional alignment in MNI
fsleyesDisplay.resetOverlays()

# Structural T1 after FLIRT + FNIRT
fsleyesDisplay.load(registered_t1_nonlin)

# Functional EPI transformed to MNI
fsleyesDisplay.load(epi_mni_path)

Per report  

Se EPI già allineata alla T1 in MNI  
The alignement between functional and anatomical images was visually assessed by overlaying a representative motion-corrected EPI volume onto the T1-weighted images previously normalized to MNI soace using FLIRT and FNIRT.  
The functional and anatomical images showed satisfactory spatial correspondance without additional registration. Therefore, an additionl funcitonal-to-anatomical registration step could be avoided, as the imaes were already aligned in standard space.  

Se EPi non allineata alla T1 MNI   
A representative motion-correctd EPI volume was registred to the subject's anatomical 1-weighted image using FSL epi_reg, which performs boundary-based rgistration usign the white matter segmentation obtained from FASt.  
The resulting EPI-to-T1 affine transformation was combined with the previously estimated nonlinear T1-ro-MNI trasnformationfrom FNIRT usign FSL applywarp. This allowed the functional image to be transfomed into MNI space and compared with the normalized anatomical image.  

Mettiamo immagine di anatomical-funcitonal alignment   
Representative motion-corrected fMRI volume overlaid on the T1-weighted anatomical image normalized to MNI152 space. The overlay was used to evaluate correspondence between the functional and structural brain boundaries and internal anatomical landmarks.
